# Lab 10: Scraping II (Pagination, Politeness, Breakage)

**DSA 405 · Week 10**

| | |
|---|---|
| **In class** | Friday, Oct 23 |
| **Take-home** | none; this week is a **P3 work session** |
| **Targets** | `books.toscrape.com`, `quotes.toscrape.com` |
| **P3 due** | Thursday, Nov 5, 11:59 PM |
| **Time** | ~30 min in class, then P3 work time |

> **No A this week.** Submit this notebook for in-class credit and spend the week on **P3**, which is due Thursday, Nov 5. Every technique in this lab is one your P3 scraper will need.

## Overview

Last week you scraped one page. This week we scrape fifty, and a few problems appear that a single page never has.

The data is split across many pages, so your code has to find the next page and know when to stop. This is called **pagination**. Fifty requests in a row are also a different situation for the website than one request, so we add some politeness: a **session** (one network connection that stays open and is reused for every request), a **rate limit** (a fixed pause between requests), and a **cache** (a saved copy of every page you have already fetched, so your code never asks the site for the same page twice).

The main subject today, though, is **failing loudly**: writing code that stops with an error message instead of continuing with wrong output. The opposite is failing silently, and for a scraper it looks like this. Page 12 returns HTTP 200 (the "success" status), but the layout of that page has changed, so your selector matches nothing. The crawl adds zero rows from that page and finishes as if nothing went wrong. You end up with a dataset that is 20 rows short, and there is no error message anywhere to tell you. That is the failure we practice catching today, because it produces a wrong dataset instead of a crash.

In [ ]:
# ---------------------------------------------------------------------------
# DSA 405 setup
# ---------------------------------------------------------------------------
import pandas as pd, numpy as np, requests, time, io
from bs4 import BeautifulSoup

# The User-Agent header tells the site who is making the request.
# Many sites refuse requests from anonymous scripts.
HEADERS = {"User-Agent": "DSA405-student-coursework (NC State; teaching exercise)"}

pd.set_option("display.width", 160)
print("pandas", pd.__version__, "| bs4 ready")

---
## Task 1: Find the "next" link, and know when to stop

You can move from page to page by guessing the URL pattern or by following the "next" link, and the two methods fail in different ways.

The first way is to guess the URL pattern: `page-1.html`, `page-2.html`, and so on. This is simple to write, but if the site changes the pattern, or you guess the last page number wrong, your code collects the wrong pages and shows no error message.

The second way is to follow the "next" link. Each page contains a link to the page after it, and the last page does not. Your code reads that link instead of guessing the URL, so it needs no page count, and it stops on its own when the link is gone. This is the way we use today.

Find the "next" link element on page 1 with Inspect (right-click the "next" button at the bottom of the page), then confirm that the same selector returns `None` on the last page.

In [ ]:
def get_soup(url, session=None):
    getter = session or requests
    r = getter.get(url, headers=HEADERS, timeout=20)
    r.raise_for_status()
    return BeautifulSoup(r.text, "html.parser")


s1 = get_soup("https://books.toscrape.com/catalogue/page-1.html")

nxt = s1.select_one("REPLACE_ME")       # find it with Inspect, the same way you found the book selectors in Lab 9
print("next link on page 1 :", nxt)

In [ ]:
# now the last page. books.toscrape.com has 50 catalogue pages, and page 51 returns a 404.
s_last = get_soup("https://books.toscrape.com/catalogue/page-50.html")
print("next link on page 50:", s_last.select_one("REPLACE_ME"))

When the "next" link is missing, the loop should stop. This is better than writing the number 50 into your code, because you do not need to know the page count in advance, and the code keeps working if the site adds a page. The missing link is not a safe stop condition on its own, though: a bug in your code could make the link look present on every page, and then the loop would never end. So the crawl in Task 2 stops when the "next" link is missing **and** stops after a maximum number of pages. The maximum is a safety limit in case the first condition never happens.

## Task 2: A session, a rate limit, and a cache

These are small habits, and each one prevents a large problem.

A session (`requests.Session()`) keeps one network connection open and reuses it for every request. Over fifty requests, this is noticeably faster than opening a new connection each time.

A rate limit is a fixed pause between requests, and we use one second. This practice site accepts fast requests, but real websites do not: the sites you will want for P3 respond with error 429 ("too many requests") or block your address completely. Add the one-second pause now, so it is already part of your habit when you work with a real site.

A cache is a saved copy of every page you have already fetched, so your code never asks the site for the same URL twice in one run. While you are debugging on a real site, re-fetching the same pages many times is one of the most common ways to get blocked.

The `fetch` function below does all three. Read it before you run it: the `CACHE` dictionary is the saved copy, `time.sleep(delay)` is the pause, and the `session` argument is the open connection.

In [ ]:
CACHE = {}

def fetch(url, session, delay=1.0):
    """Fetch each URL at most once, wait between requests, and save the result."""
    if url in CACHE:
        print(f"  cached: {url.split('/')[-1]}")
        return CACHE[url]
    r = session.get(url, headers=HEADERS, timeout=20)
    r.raise_for_status()
    CACHE[url] = r.text
    time.sleep(delay)
    return r.text


with requests.Session() as sess:
    base = "https://books.toscrape.com/catalogue/page-{}.html"
    for n in [1, 2, 1]:            # page 1 appears twice on purpose, so you can see the cache working
        html = fetch(base.format(n), sess)
        print(f"page {n}: {len(html)} chars")

The third fetch printed `cached:`, so the page came from the saved copy in memory instead of being requested again. That is exactly what you want while debugging.

Now the **crawl**, which is a loop that scrapes many pages one after another. Scrape the first **five** catalogue pages into one DataFrame by following the "next" link, with a maximum-pages safety limit. The book selectors are the ones you found in Lab 9. Decide what you expect before you run: 100 rows, 20 from each page, and no warnings.

In [ ]:
def crawl(start_url, max_pages=5, expected_per_page=20):
    """Follow 'next' links, collecting books, with a maximum page count.

    Returns (DataFrame, list of warnings). A warning records a page that did
    not look the way we expected, without stopping the crawl.
    """
    rows, warnings, url, pages = [], [], start_url, 0

    with requests.Session() as sess:
        while url and pages < max_pages:
            html = fetch(url, sess)
            soup = BeautifulSoup(html, "html.parser")
            found = soup.select("REPLACE_ME")

            if len(found) != expected_per_page:
                warnings.append(f"{url}: expected {expected_per_page}, found {len(found)}")

            for b in found:
                rows.append({"title": ..., "price_raw": ..., "page": pages + 1})

            pages += 1
            nxt = soup.select_one("REPLACE_ME")
            url = requests.compat.urljoin(url, nxt["href"]) if nxt else None

    return pd.DataFrame(rows), warnings


books, warns = crawl("https://books.toscrape.com/catalogue/page-1.html", max_pages=5)
print(f"rows: {len(books)}   expected: 100")
print(f"warnings: {warns or 'none'}")
books.groupby("page").size()

## Task 3: Break it on purpose

This is the most important task in the lab, and it is the reason the lab exists.

The most dangerous way for a scraper to fail is to keep running and produce wrong output. Suppose page 12 returns HTTP 200 with a changed layout. The selector matches nothing, `select` returns an empty list, and the loop appends zero rows for that page. The crawl finishes, reports success, and the dataset is 20 rows short with no error message anywhere.

We simulate this below. The crawl runs with a deliberately wrong selector on one page (`"article.nonexistent"` on page 2), and you watch what each version of the code does with it. The first version has no checks at all, which is what most first drafts of a scraper look like.

In [ ]:
def crawl_silent(start_url, max_pages=3, break_page=2):
    """A crawl with NO checks. This is what most first drafts look like."""
    rows, url, pages = [], start_url, 0
    with requests.Session() as sess:
        while url and pages < max_pages:
            soup = BeautifulSoup(fetch(url, sess), "html.parser")
            sel = "article.nonexistent" if pages + 1 == break_page else "REPLACE_ME"
            for b in soup.select(sel):
                rows.append({"title": ..., "page": pages + 1})
            pages += 1
            nxt = soup.select_one("REPLACE_ME")
            url = requests.compat.urljoin(url, nxt["href"]) if nxt else None
    return pd.DataFrame(rows)


quiet = crawl_silent("https://books.toscrape.com/catalogue/page-1.html")
print(f"rows returned: {len(quiet)}   (should be 60)")
print("nothing warned you. That's the problem.")
print()
print(quiet.groupby("page").size())

Now write the version that catches the problem. Each response in the table below is right in a different situation.

| Level | Behavior | Use when |
|---|---|---|
| **Warn** | record a warning message, keep going | a partial dataset is still useful |
| **Raise** | stop the crawl with an error | a partial dataset would be misleading |
| **Assert at the end** | check the total row count | the expected total is known in advance |

Write a crawl that does all three, and show it catching the broken page. Start by copying `crawl_silent` into the cell below and keep the deliberately wrong selector on page 2, because you need a broken page in order to show the catch. For the warning, a list like the `warnings` list in `crawl` from Task 2 is enough. For the raise, use `raise ValueError(...)` with a message that names the page. For the assert, compare the row count against the number you expected before you ran the crawl.

In [ ]:
# your crawl that fails loudly instead of silently

## Task 4: A different site, a different shape

Selectors do not transfer between sites, so scraping a second site is the fastest way to find out which parts of your approach work anywhere and which parts only work on `books.toscrape.com`. Scrape the first **three** pages of `quotes.toscrape.com` and extract the quote text, the author, and the tags. Find the selectors with Inspect, the same way you did in Lab 9, and decide the number of quotes you expect per page before you run the loop.

The tags make you decide the **grain** of your table, which means deciding what one row represents. Each quote has several tags, so you must choose: one row per quote, with the tags stored as a list, or one row per quote-tag pair? This is the grain problem from Week 6.

In [ ]:
# scrape quotes.toscrape.com

State the grain you chose, and why. Then describe one question that the other grain would answer more easily.

*Answer here.*

---
## Checkpoint: submit before leaving

1. Which pagination method did you use, and what safety limit stops the loop if the "next" link never disappears?
2. How many rows did `crawl_silent` return, and how many should it have returned?
3. For your P3 scraper: should it warn, raise, or assert when a page returns no rows? Explain why you chose that response for your source.
4. What grain did you choose for the quote tags, and how many rows did your table have?
5. One sentence about P3: what is the most likely way your scraper could fail between now and Nov 5?

*Answers here.*

---
## Before leaving

**P3 is due Thursday, Nov 5**, after next week's API session. It needs the acquisition code, the join, and a verification suite with at least five assertions, plus a demonstration of the suite catching three mutated copies of your own joined data. Everything you wrote today applies to the acquisition code: a cache, a pause between requests, pagination if your source has it, and one demonstrated catch of a condition that would otherwise produce wrong output with no error message.

**Bench Check 2 runs during the lab block in Weeks 10, 11, and 12.** If you do not have a slot yet, sign up today on Moodle under Resources.

If a source has stopped working since P1, email me within 48 hours. Adapting to a broken source, and documenting how you adapted, counts as a strength on the P3 rubric, and telling me early gives us time to fix the plan together before the deadline.

---
## AI use note

Tell me which AI tools you used here and what you used them for, in a sentence or two.
If you didn't use any, write "none."

*Answer here.*

---
## Submitting

1. **Runtime > Restart runtime**, then **Run all**.
2. `File > Download > Download .ipynb`
3. Upload the notebook to **Week 10 In-Class Activity** on Moodle before the end of class on Friday. There is no take-home this week, so there is nothing to rename and no A dropbox.